## Tutorial 1. ResNet18 on CIFAR10. 


In this tutorial, we will show 

- How to end-to-end train and compress a ResNet18 from scratch on CIFAR10 to get a compressed ResNet18.
- The compressed ResNet18 achives both **high performance** and **significant FLOPs and parameters reductions** than the full model. 
- The compressed ResNet18 **reduces about 92% parameters** to achieve **92.91% accuracy** only lower than the baseline by **0.11%**.
- More detailed new HESSO optimizer setup. (Technical report regarding HESSO will be released on the early of 2024).

### Step 1. Create OTO instance

In [1]:
import torch

from only_train_once import OTO
from sanity_check.backends.resnet_cifar10 import resnet18_cifar10

model = resnet18_cifar10()
dummy_input = torch.rand(1, 3, 32, 32)
oto = OTO(model=model.cuda(), dummy_input=dummy_input.cuda())

/home/xiaoyi/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OTO graph constructor
graph build


#### (Optional) Visualize the pruning dependancy graph of DNN

In [2]:
# A ResNet_zig.gv.pdf will be generated to display the depandancy graph.
oto.visualize(view=False, out_dir="../cache")

### Step 2. Dataset Preparation

In [3]:
from torchvision import transforms
from torchvision.datasets import CIFAR10

trainset = CIFAR10(
    root="cifar10",
    train=True,
    download=True,
    transform=transforms.Compose(
        [
            transforms.RandomHorizontalFlip(),
            transforms.RandomCrop(32, 4),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    ),
)
testset = CIFAR10(
    root="cifar10",
    train=False,
    download=True,
    transform=transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    ),
)

trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=64, shuffle=True, num_workers=4
)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=64, shuffle=False, num_workers=4
)

Files already downloaded and verified
Files already downloaded and verified


### Step 3. Setup HESSO optimizer

The following main hyperparameters need to be taken care.

- `variant`: The optimizer that is used for training the baseline full model. Currently support `sgd`, `adam` and `adamw`.
- `lr`: The initial learning rate.
- `weight_decay`: Weight decay as standard DNN optimization.
- `target_group_sparsity`: The target group sparsity, typically higher group sparsity refers to more FLOPs and model size reduction, meanwhile may regress model performance more.
- `start_pruning_steps`: The number of steps that **starts** to prune.
- `pruning_steps`: The number of steps that **finishes** pruning (reach `target_group_sparsity`) after `start_pruning_steps`.
- `pruning_periods`:  Incrementally produce the group sparsity equally among pruning periods.

We empirically suggest `start_pruning_steps` as 1/10 of total number of training steps. `pruning_steps` until 1/4 or 1/5 of total number of training steps.
The advatnages of HESSO compared to DHSPG is its explicit control over group sparsity exploration, which is typically more convenient.

In [4]:
optimizer = oto.hesso(
    variant="sgd",
    lr=0.1,
    weight_decay=1e-4,
    target_group_sparsity=0.7,
    start_pruning_step=10 * len(trainloader),
    pruning_periods=10,
    pruning_steps=10 * len(trainloader),
)

Setup HESSO
Target redundant groups per period:  [201, 201, 201, 201, 201, 201, 201, 201, 201, 206]


### Step 4. Train ResNet18 as normal.

In [5]:
from utils.utils import check_accuracy

max_epoch = 100
model.cuda()
criterion = torch.nn.CrossEntropyLoss()
# Every 50 epochs, decay lr by 10.0
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.1)

for epoch in range(max_epoch):
    f_avg_val = 0.0
    model.train()
    lr_scheduler.step()
    for X, y in trainloader:
        X = X.cuda()
        y = y.cuda()
        y_pred = model.forward(X)
        f = criterion(y_pred, y)
        optimizer.zero_grad()
        f.backward()
        f_avg_val += f
        optimizer.step()
    opt_metrics = optimizer.compute_metrics()
    # group_sparsity, param_norm, _ = optimizer.compute_group_sparsity_param_norm()
    # norm_important, norm_redundant, num_grps_important, num_grps_redundant = optimizer.compute_norm_groups()
    accuracy1, accuracy5 = check_accuracy(model, testloader)
    f_avg_val = f_avg_val.cpu().item() / len(trainloader)

    print(
        f"Ep: {epoch}, loss: {f_avg_val:.2f}, norm_all:{opt_metrics.norm_params:.2f}, grp_sparsity: {opt_metrics.group_sparsity:.2f}, acc1: {accuracy1:.4f}, norm_import: {opt_metrics.norm_important_groups:.2f}, norm_redund: {opt_metrics.norm_redundant_groups:.2f}, num_grp_import: {opt_metrics.num_important_groups}, num_grp_redund: {opt_metrics.num_redundant_groups}"
    )

/home/xiaoyi/miniconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:143: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn("Detected call of `lr_scheduler.step()` before `optimizer.step()`. "


/home/xiaoyi/miniconda3/lib/python3.12/site-packages/torch/nn/modules/conv.py:456: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at /opt/conda/conda-bld/pytorch_1712609048481/work/aten/src/ATen/native/cudnn/Conv_v8.cpp:919.)
  return F.conv2d(input, weight, bias, self.stride,


Ep: 0, loss: 1.61, norm_all:4123.36, grp_sparsity: 0.00, acc1: 0.3796, norm_import: 4123.36, norm_redund: 0.00, num_grp_import: 2880, num_grp_redund: 0
Ep: 1, loss: 1.06, norm_all:4114.36, grp_sparsity: 0.00, acc1: 0.4631, norm_import: 4114.36, norm_redund: 0.00, num_grp_import: 2880, num_grp_redund: 0
Ep: 2, loss: 0.81, norm_all:4105.85, grp_sparsity: 0.00, acc1: 0.7201, norm_import: 4105.85, norm_redund: 0.00, num_grp_import: 2880, num_grp_redund: 0
Ep: 3, loss: 0.66, norm_all:4095.98, grp_sparsity: 0.00, acc1: 0.7359, norm_import: 4095.98, norm_redund: 0.00, num_grp_import: 2880, num_grp_redund: 0
Ep: 4, loss: 0.57, norm_all:4085.42, grp_sparsity: 0.00, acc1: 0.7351, norm_import: 4085.42, norm_redund: 0.00, num_grp_import: 2880, num_grp_redund: 0
Ep: 5, loss: 0.50, norm_all:4074.24, grp_sparsity: 0.00, acc1: 0.7689, norm_import: 4074.24, norm_redund: 0.00, num_grp_import: 2880, num_grp_redund: 0
Ep: 6, loss: 0.45, norm_all:4062.65, grp_sparsity: 0.00, acc1: 0.8144, norm_import: 4062

### Step 5. Get compressed model in torch format

In [6]:
# By default OTO will construct subnet by the last checkpoint. If intermedia ckpt reaches the best performance,
# need to reinitialize OTO instance
# oto = OTO(torch.load(ckpt_path), dummy_input)
# then construct subnetwork
oto.construct_subnet(out_dir="./cache")

### (Optional) Check the compressed model size

In [7]:
import os

full_model_size = os.stat(oto.full_group_sparse_model_path)
compressed_model_size = os.stat(oto.compressed_model_path)
print("Size of full model     : ", full_model_size.st_size / (1024**3), "GBs")
print("Size of compress model : ", compressed_model_size.st_size / (1024**3), "GBs")

Size of full model     :  0.041716959327459335 GBs
Size of compress model :  0.003497043624520302 GBs


### (Optional) Check the compressed model accuracy
#### # Both full and compressed model should return the exact same accuracy.

In [8]:
full_model = torch.load(oto.full_group_sparse_model_path)
compressed_model = torch.load(oto.compressed_model_path)

acc1_full, acc5_full = check_accuracy(full_model, testloader)
print(f"Full model: Acc 1: {acc1_full}, Acc 5: {acc5_full}")

acc1_compressed, acc5_compressed = check_accuracy(compressed_model, testloader)
print(f"Compressed model: Acc 1: {acc1_compressed}, Acc 5: {acc5_compressed}")

Full model: Acc 1: 0.9269, Acc 5: 0.9971
Compressed model: Acc 1: 0.9269, Acc 5: 0.9971
